# Fine-tune Gemma 4 E4B for NCD Risk Prediction

This notebook fine-tunes Gemma 4 E4B on synthetic EHR data to predict diabetes and hypertension risk.

**Requirements:**
- Lightning AI with H200 GPU (80GB VRAM)
- ~30-45 minutes training time (full dataset)

**Dataset:** `samwell/synthea-ncd-instructions`

**Optimized settings for H200 (80GB VRAM)**

## 1. Setup Environment

In [1]:
# Check GPU
!nvidia-smi

In [2]:
# Install Unsloth (optimized for Colab)
# This may take 2-3 minutes
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes xformers

In [ ]:
# Imports
import os
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import gc

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
# Model config - H200 OPTIMIZED (80GB VRAM)
MODEL_NAME = "google/gemma-4-E4B-it"  # 4B effective - plenty of room on H200
MAX_SEQ_LENGTH = 2048  # Full context length
LOAD_IN_4BIT = True  # QLoRA still for efficiency

# LoRA config
LORA_R = 32  # Higher rank for better quality
LORA_ALPHA = 32
LORA_DROPOUT = 0

# Training config - OPTIMIZED FOR H200
BATCH_SIZE = 8           # Large batch for H200
GRADIENT_ACCUMULATION = 2  # Effective batch = 16
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
WARMUP_STEPS = 100

# Dataset - FULL DATASET (H200 can handle it)
DATASET_NAME = "samwell/synthea-ncd-instructions"
MAX_SAMPLES = None  # Use full dataset

# Output
OUTPUT_DIR = "./ncd-predictor-checkpoints"
MODEL_OUTPUT = "./ncd-predictor-lora"

print("H200 optimized config (80GB VRAM):")
print(f"  Model: {MODEL_NAME}")
print(f"  Batch: {BATCH_SIZE} x {GRADIENT_ACCUMULATION} = {BATCH_SIZE * GRADIENT_ACCUMULATION} effective")
print(f"  Seq length: {MAX_SEQ_LENGTH}")
print(f"  LoRA rank: {LORA_R}")
print(f"  Samples: Full dataset")

## 3. Load Dataset

In [ ]:
# Load from HuggingFace
dataset = load_dataset(DATASET_NAME)

# Check available splits
print(f"Available splits: {list(dataset.keys())}")

# Handle different split naming
train_split = "train"
val_split = "val" if "val" in dataset else "validation" if "validation" in dataset else None
test_split = "test" if "test" in dataset else None

print(f"Full train examples: {len(dataset[train_split])}")

# Optionally limit samples (set MAX_SAMPLES = None for full dataset)
if MAX_SAMPLES and len(dataset[train_split]) > MAX_SAMPLES:
    dataset[train_split] = dataset[train_split].shuffle(seed=42).select(range(MAX_SAMPLES))
    print(f"Limited to: {MAX_SAMPLES} samples")
else:
    print("Using FULL dataset")

print(f"Training on: {len(dataset[train_split])} examples")
if val_split:
    print(f"Validation examples: {len(dataset[val_split])}")
if test_split:
    print(f"Test examples: {len(dataset[test_split])}")

In [6]:
# View a sample
sample = dataset['train'][0]
print("=" * 60)
print("INSTRUCTION:")
print(sample['instruction'][:200] + "...")
print("\nINPUT:")
print(sample['input'])
print("\nOUTPUT:")
print(sample['output'][:500] + "...")

## 4. Load Model with QLoRA

In [7]:
# Load base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # Auto-detect
)

print(f"Model loaded: {MODEL_NAME}")

In [ ]:
# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Print trainable parameters
def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

trainable, total = count_parameters(model)
print(f"Trainable parameters: {trainable:,} ({100*trainable/total:.2f}%)")
print(f"Total parameters: {total:,}")

## 5. Format Prompts

In [ ]:
# Gemma chat template
def format_prompt(examples):
    """Format examples into Gemma chat format. Returns a list of strings."""
    outputs = []
    for instruction, input_text, output_text in zip(
        examples['instruction'], 
        examples['input'], 
        examples['output']
    ):
        text = f"""<start_of_turn>user
{instruction}

{input_text}<end_of_turn>
<start_of_turn>model
{output_text}<end_of_turn>"""
        outputs.append(text)
    return outputs

# Test formatting
test_batch = dataset['train'][:2]
formatted = format_prompt(test_batch)
print(formatted[0][:800])

## 6. Train

In [ ]:
# Training arguments - H200 optimized
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
)

# Calculate expected training time
train_size = len(dataset['train'])
steps_per_epoch = train_size // (BATCH_SIZE * GRADIENT_ACCUMULATION)
total_steps = steps_per_epoch * NUM_EPOCHS
print(f"Training samples: {train_size}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Total steps: {total_steps}")
print(f"Estimated time: ~{total_steps / 3.0 / 60:.0f} minutes (H200 ~3 it/s)")

In [ ]:
# Create trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    formatting_func=format_prompt,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

print("Trainer created successfully!")

In [ ]:
# Start training
print("Starting training on H200...")
print(f"Model: {MODEL_NAME}")
print(f"Samples: {len(dataset['train'])}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Total steps: {total_steps}")
print("-" * 50)

trainer_stats = trainer.train()

print("\n" + "=" * 50)
print("Training complete!")
print(f"Final loss: {trainer_stats.training_loss:.4f}")
print(f"Total time: {trainer_stats.metrics['train_runtime'] / 60:.1f} minutes")

## 7. Save Model

In [ ]:
# Save LoRA adapters
model.save_pretrained(MODEL_OUTPUT)
tokenizer.save_pretrained(MODEL_OUTPUT)

print(f"Model saved to: {MODEL_OUTPUT}")

In [ ]:
# Optional: Push to HuggingFace Hub
PUSH_TO_HUB = False  # Set to True to push
HUB_MODEL_NAME = "samwell/ncd-predictor-gemma4-lora"

if PUSH_TO_HUB:
    from huggingface_hub import login
    login()  # Will prompt for token
    
    model.push_to_hub(HUB_MODEL_NAME)
    tokenizer.push_to_hub(HUB_MODEL_NAME)
    print(f"Pushed to: https://huggingface.co/{HUB_MODEL_NAME}")

## 8. Test Inference

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

# Test prompt
test_input = """Patient: 55yo Male
Vitals: BP 152/94 mmHg, BMI 31.2, Weight 92 kg, HR 78 bpm
Labs: Glucose 128 mg/dL, HbA1c 6.4%, Total Cholesterol 245 mg/dL, LDL 165 mg/dL
Active conditions: None
Medications: None"""

prompt = f"""<start_of_turn>user
Based on the following patient record, assess the risk of Type 2 diabetes and hypertension. Provide risk levels (LOW, MODERATE, HIGH, or DIAGNOSED) with supporting factors, and clinical recommendations.

{test_input}<end_of_turn>
<start_of_turn>model
"""

# Use text= parameter explicitly
inputs = tokenizer(text=prompt, return_tensors="pt").to("cuda")

# Generate
outputs = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

# Decode
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response.split("<start_of_turn>model")[-1])

## 9. Export to GGUF (Optional)

For deployment on mobile/edge devices with llama.cpp

In [ ]:
EXPORT_GGUF = False  # Set to True to export

if EXPORT_GGUF:
    # Merge LoRA weights
    model.save_pretrained_merged(
        "./ncd-predictor-merged",
        tokenizer,
        save_method="merged_16bit",
    )
    
    # Export to GGUF (requires llama.cpp)
    model.save_pretrained_gguf(
        "./ncd-predictor-gguf",
        tokenizer,
        quantization_method="q4_k_m",
    )
    
    print("GGUF exported!")

## 10. Download Model

Download the trained model to your local machine.

In [ ]:
# Zip model for download
!zip -r ncd-predictor-lora.zip ./ncd-predictor-lora

print("Model zipped! Download from Lightning AI file browser or use:")
print("  lightning cp ./ncd-predictor-lora.zip /teamspace/studios/this_studio/")

---

## Summary

You've fine-tuned Gemma 4 E4B for NCD risk prediction!

**Training Config (H200 optimized):**
- Full dataset: 39,371 samples
- Model: Gemma 4 E4B (4B effective params)
- Batch size 8 × 2 gradient accumulation = 16 effective
- Sequence length 2048
- LoRA rank 32
- ~30-45 minutes on H200

**Next steps:**
1. Evaluate on test set
2. Validate with clinical experts
3. Deploy (HuggingFace, GGUF for mobile, or API)

**Resources:**
- Dataset: https://huggingface.co/datasets/samwell/synthea-ncd-instructions
- Unsloth docs: https://unsloth.ai/docs